In [0]:
jdbc_url = "jdbc:postgresql://0.tcp.in.ngrok.io:29031/demo"

username = "postgres"
password = "root"

driver = "org.postgresql.Driver"

# Getting last watermark

In [0]:
control = spark.table("migration.control.ingestion_state")

last_watermark = (
    control
    .filter(
        (control.source_system == "PostgreSQL") &
        (control.source_table == "customers") &
        (control.last_status == "SUCCESS")
    )
    .select("last_watermark")
    .collect()[0][0]
)

print("Last successful watermark:", last_watermark)

# incremental batch query

In [0]:
incremental_query = f"""
SELECT *
FROM public.customers
WHERE updated_at > '{last_watermark}'
"""

In [0]:
incremental_df = (
    spark.read
    .format("jdbc")
    .option("driver", driver)
    .option("url", jdbc_url)
    .option("query", incremental_query)
    .option("user", username)
    .option("password", password)
    .load()
)

display(incremental_df)

# New batch generation

In [0]:
from datetime import datetime
import uuid

batch_id = f"MIGRATION_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_id = str(uuid.uuid4())

print("Batch ID :", batch_id)
print("Run ID   :", run_id)

### new watermark

In [0]:
new_watermark = (
    incremental_df
    .agg({"updated_at": "max"})
    .collect()[0][0]
)

print("New watermark:", new_watermark)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

incremental_bronze_df = (
    incremental_df
    .withColumn("_batch_id", lit(batch_id))
    .withColumn("_run_id", lit(run_id))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_system", lit("PostgreSQL"))
    .withColumn("_source_table", lit("customers"))
)
display(incremental_bronze_df)

# Writing in bronze

In [0]:
(
    incremental_bronze_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("migration.bronze.customers")
)

In [0]:
%sql
SELECT
    _batch_id,
    _run_id,
    COUNT(*) AS row_count
FROM migration.bronze.customers
GROUP BY _batch_id, _run_id
ORDER BY _batch_id;

In [0]:
last_row_count = (
    incremental_df
    .count()
)

display(last_row_count)

# Update the control table

In [0]:
from pyspark.sql.functions import current_timestamp

control_update_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            new_watermark,
            batch_id,
            run_id,
            "SUCCESS",
            last_row_count
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_row_count BIGINT
    """
).withColumn(
    "last_processed_at",
    current_timestamp()
)

control_update_df.createOrReplaceTempView("control_update")

# Merge

In [0]:
%sql
MERGE INTO migration.control.ingestion_state AS target
USING control_update AS source
ON  target.source_system = source.source_system
AND target.source_table = source.source_table

WHEN MATCHED THEN UPDATE SET
    target.last_watermark = source.last_watermark,
    target.last_batch_id = source.last_batch_id,
    target.last_run_id = source.last_run_id,
    target.last_status = source.last_status,
    target.last_processed_at = source.last_processed_at,
    target.last_row_count = source.last_row_count

WHEN NOT MATCHED THEN INSERT (
    source_system,
    source_table,
    last_watermark,
    last_batch_id,
    last_run_id,
    last_status,
    last_processed_at,
    last_row_count
)
VALUES (
    source.source_system,
    source.source_table,
    source.last_watermark,
    source.last_batch_id,
    source.last_run_id,
    source.last_status,
    source.last_processed_at,
    source.last_row_count
);

In [0]:
%sql
SELECT *
FROM migration.control.ingestion_state;

# ---------Failure Testing----------------

In [0]:
%sql
CREATE TABLE IF NOT EXISTS migration.control.ingestion_log (
    source_system STRING,
    source_table STRING,
    batch_id STRING,
    run_id STRING,
    start_watermark TIMESTAMP,
    end_watermark TIMESTAMP,
    status STRING,
    row_count BIGINT,
    batch_started_at TIMESTAMP,
    batch_completed_at TIMESTAMP
)
USING DELTA;

In [0]:
batch4_end_watermark = (
    incremental_df
    .agg({"updated_at": "max"})
    .collect()[0][0]
)

print("Batch 4 end watermark:", batch4_end_watermark)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

failed_log_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            "MIGRATION_20260825_150152",
            "2c4e55f4-c7cc-4911-86eb-ee1abfc8daa8",
            last_watermark,
            batch4_end_watermark,
            "FAILED",
            2
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    batch_id STRING,
    run_id STRING,
    start_watermark TIMESTAMP,
    end_watermark TIMESTAMP,
    status STRING,
    row_count BIGINT
    """
).withColumn(
    "batch_started_at",
    current_timestamp()
).withColumn(
    "batch_completed_at",
    current_timestamp()
)

In [0]:
(
    failed_log_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("migration.control.ingestion_log")
)

In [0]:
%sql
SELECT *
FROM migration.control.ingestion_log;

In [0]:
%sql
SELECT
    _batch_id,
    COUNT(*) AS row_count
FROM migration.bronze.customers
WHERE _batch_id = 'MIGRATION_20260825_150152'
GROUP BY _batch_id;

In [0]:
existing_batch = spark.sql("""
    SELECT COUNT(*) AS cnt
    FROM migration.bronze.customers
    WHERE _batch_id = 'MIGRATION_20260825_150152'
""").collect()[0]["cnt"]

print("Existing Bronze rows for Batch 4:", existing_batch)

In [0]:
import uuid

retry_run_id = str(uuid.uuid4())

print("Retry Run ID:", retry_run_id)

In [0]:
from pyspark.sql.functions import current_timestamp

retry_log_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            "MIGRATION_20260825_150152",
            retry_run_id,
            last_watermark,
            batch4_end_watermark,
            "SUCCESS",
            2
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    batch_id STRING,
    run_id STRING,
    start_watermark TIMESTAMP,
    end_watermark TIMESTAMP,
    status STRING,
    row_count BIGINT
    """
).withColumn(
    "batch_started_at",
    current_timestamp()
).withColumn(
    "batch_completed_at",
    current_timestamp()
)

In [0]:
(
    retry_log_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("migration.control.ingestion_log")
)

In [0]:
control_retry_df = spark.createDataFrame(
    [
        (
            "PostgreSQL",
            "customers",
            batch4_end_watermark,
            "MIGRATION_20260825_150152",
            retry_run_id,
            "SUCCESS",
            2
        )
    ],
    """
    source_system STRING,
    source_table STRING,
    last_watermark TIMESTAMP,
    last_batch_id STRING,
    last_run_id STRING,
    last_status STRING,
    last_row_count BIGINT
    """
).withColumn(
    "last_processed_at",
    current_timestamp()
)

control_retry_df.createOrReplaceTempView("control_retry")

In [0]:
%sql
MERGE INTO migration.control.ingestion_state AS target
USING control_retry AS source
ON  target.source_system = source.source_system
AND target.source_table = source.source_table

WHEN MATCHED THEN UPDATE SET
    target.last_watermark = source.last_watermark,
    target.last_batch_id = source.last_batch_id,
    target.last_run_id = source.last_run_id,
    target.last_status = source.last_status,
    target.last_processed_at = source.last_processed_at,
    target.last_row_count = source.last_row_count

WHEN NOT MATCHED THEN INSERT (
    source_system,
    source_table,
    last_watermark,
    last_batch_id,
    last_run_id,
    last_status,
    last_processed_at,
    last_row_count
)
VALUES (
    source.source_system,
    source.source_table,
    source.last_watermark,
    source.last_batch_id,
    source.last_run_id,
    source.last_status,
    source.last_processed_at,
    source.last_row_count
);

In [0]:
%sql
SELECT
    source_table,
    last_watermark,
    last_batch_id,
    last_run_id,
    last_status,
    last_row_count
FROM migration.control.ingestion_state;